# Getting started

The library's core concept is the `Mesh` class which holds tensors with the mesh information
(vertex locations, element definitions), as well as additional features that are defined at
the vertex, element, or mesh level.

In more details, the `Mesh` attributes are 

| Field | type | tensor(s) shape | Description |
|---|---|---|---|
| `xy` | Tensor (float) | (num_vertex, 2) | Vertex coordinates |
| `cell_indices` | Tensor (int64) | (num_cells, 3) | vertex indices for each element |
| `vertex_features` | frozendict[str, Tensor] | (num_vertex, ...) | Per-vertex features |
| `cell_features` | frozendict[str, Tensor]| (num_cells, ...) | Per-cell features |
| `global_features` | frozendict[str, Tensor] | (...) | Mesh-level attributes |


In [ ]:
from __future__ import annotations

from pathlib import Path

import rich
import torch

from tensormesh import Mesh, plots

%load_ext autoreload
%autoreload 2

## 1. Creating or loading a Mesh

`Mesh` object can be instanciated from scratch by specifying the mesh vertices location and the triangular element indices. A very simple example of that is

In [ ]:
simple_mesh = Mesh(
    xy=torch.tensor([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0]]),
    cell_indices=torch.tensor([[0, 1, 2], [0, 2, 3]]),
)

rich.print(simple_mesh)

plots.plot_mesh(simple_mesh)

In this tutorial, we will however focus on a more interesting example: the 2D mesh of the cross section of an electric motor which has been saved alongside this tutorial. This example mesh has a rich per-cell and per-vertex physics features.

In [ ]:
# loading and saving of mesh objects is done via torch.save and torch.load.
mesh = torch.load(Path("motor_mesh.pt"), weights_only=False)

# Print some information about the mesh.
print(f"Num vertices : {mesh.num_vertices}")
print(f"Num cells    : {mesh.num_cells}")
print(f"Device       : {mesh.device}")

## 2. Accessing the content of a Mesh

All tensors attributes are accessible as plain `torch.Tensor`. For the features, these tensors are stored in dictionaries.

In [ ]:
# vertex positions
print(f"vertex positions: {mesh.xy}\n")

# A-field is scalar defined at vertices. It is stored in the vertex_feature dictionary:
print(f"A-field: {mesh.vertex_features['magnetic vector potential']}\n")

# B-field is a vector defined at cells. It is stored in the cell_feature dictionary:
print(f"B-field: {mesh.cell_features['magnetic flux density']}\n")

## 3. Operations

`tensormesh.ops` provides stateless convenience functions that operate directly on tensors.
The most notable ones are

In [ ]:
from tensormesh.ops import cell_areas, interpolate_at_cells

# Cell areas via the 2D cross product
areas = cell_areas(mesh.xy, mesh.cell_indices)
print(f"Range of cell areas: [{areas.min():.2e}, {areas.max():.2e}]")
print(f"Total mesh area : {areas.sum():.4f}")

# Vertex-to-cell interpolation (mean of the three corner values)
# This can for be used to compute the cell barycenters
xy_cell = interpolate_at_cells(mesh.xy, mesh.cell_indices)
print(f"Cell barycenters shape: {xy_cell.shape}")

## 4. Plotting

All plot methods return a Plotly `Figure`.
Pass a column name from any schema field to colour cells or vertices:
boolean flags produce categorical colouring; numeric fields use a continuous colour scale.

`plot_materials()` is a convenience method that decodes all one-hot `cell_flags` columns at once
into per-cell material labels and produces a categorical plot — one trace (and one legend entry)
per material.

In [ ]:
# Plain mesh geometry
plots.plot_mesh(mesh)

In [ ]:
# plot a continuous value at cells
fig = plots.plot_cell_features(mesh, "current density")
fig.show()

# plot a boolean value at cells
fig = plots.plot_cell_features(mesh, "steel")
fig.show()

In [ ]:
material_columns = ["steel", "air", "copper", "magnet"]

plots.plot_boolean_cell_features(mesh, material_columns)

In [ ]:
# Vertex feature plotted as a coloured scatter on top of the mesh cells
plots.plot_vertex_features(mesh, "magnetic vector potential", show_cells=True)

In [ ]:
# 3D surface using a vertex column as the z-coordinate
plots.plot_wireframe(mesh, "magnetic vector potential")

In [ ]:
# Magnetic flux density B = (Bx, By) as quiver arrows at cell centroids,
# coloured by the is_copper flag (adjust scale to taste)

material_colormap = {
    "shaft": "#FFA15A",
    "steel": "#636EFA",
    "anisotropic steel": "#636EFA",
    "copper": "#EF553B",
    "air": "#FECB52",
    "magnet": "#00CC96",
}

plots.plot_vector_field(
    mesh,
    face_type="cell",
    features="magnetic flux density",
    mask_columns=material_columns,
    scale=0.001,
    colormap=material_colormap,
)